# Gathering HLS4ML-reports


In [ ]:
import hls4ml 

search_dir = '../'

patterns = { 
    "All HLS4ML-projects"  : "hls4ml_prj*/",
    #"MNIST HLS4ML-projects" : "MNIST/*/hls4ml_prj*/",  
    #"SVHN HLS4ML-projects"  : "SVHN/*/hls4ml_prj*/",  
}

save_reports = 'reports/'

In [33]:
import os
from pathlib import Path
import sys

#os.listdir('../', )
reports = {}

for label,pattern in patterns.items():
    print(f"Searching {label} ({pattern})")
    for path in Path(search_dir).rglob(pattern):
        if path.is_dir():
            # https://docs.python.org/3/library/pathlib.html#pathlib.PurePath
            #path = path.absolute()
            project_name = f"{path.parts[-3]}-{path.parts[-2]}-{path.name}" # Dataset - Modellarkitektur - hva HLS4ML-prosjektet heter
            print(f"\n\n%%%%%%% {project_name} found in {path} %%%%%%%%%%%\n")
            report_path = f"{save_reports}/{project_name}.rpt"
            with open(report_path, 'w') as f:
                sys.stdout = f
                hls4ml.report.read_vivado_report(str(path), full_report=True) # https://github.com/fastmachinelearning/hls4ml/blob/main/hls4ml/report/vivado_report.py
                sys.stdout = sys.__stdout__
                # parse report makes a "summary"
            summary = hls4ml.report.parse_vivado_report(str(path))
            reports.update({
                project_name : {
                    'path' : str(path.absolute()),
                    'summary' : summary,
                    'report_path' : str(report_path)
                    }
                })

In [34]:
display(reports)

{'MNIST-Qkeras-hls4ml_prj_stream_latency': {'path': '/home/ncgadmin/DAT255/DAT255-project/Reports/../MNIST/Qkeras/hls4ml_prj_stream_latency',
  'summary': {'CSynthesisReport': {'TargetClockPeriod': '5.00',
    'EstimatedClockPeriod': '3.646',
    'BestLatency': '2367',
    'WorstLatency': '2367',
    'IntervalMin': '2352',
    'IntervalMax': '2352',
    'BRAM_18K': '134',
    'DSP': '46',
    'FF': '34445',
    'LUT': '143665',
    'URAM': '0',
    'AvailableBRAM_18K': '288',
    'AvailableDSP': '1248',
    'AvailableFF': '234240',
    'AvailableLUT': '117120',
    'AvailableURAM': '64'}},
  'report_path': 'reports//MNIST-Qkeras-hls4ml_prj_stream_latency.rpt'},
 'MNIST-Qkeras-hls4ml_prj_base': {'path': '/home/ncgadmin/DAT255/DAT255-project/Reports/../MNIST/Qkeras/hls4ml_prj_base',
  'summary': {'CSynthesisReport': {'TargetClockPeriod': '5.00',
    'EstimatedClockPeriod': '3.646',
    'BestLatency': '2367',
    'WorstLatency': '2367',
    'IntervalMin': '2352',
    'IntervalMax': '2352'

Extract summary to table

In [35]:
import pandas as pd

summary_fields = ['CSynthesisReport.BestLatency', 'CSynthesisReport.LUT', 'CSynthesisReport.FF', 'CSynthesisReport.DSP', 'CSynthesisReport.BRAM_18K' ]

base_columns = ["project_name"]#, "path", "report_path"]
summary_rows = [
    {
        "project_name": project_name,
        #"path": report["path"],
        #"report_path": report["report_path"],
        **report["summary"],
    }
    for project_name, report in reports.items()
]

results_table = pd.json_normalize(summary_rows, sep=".")
summary_columns = [field for field in summary_fields if field in results_table.columns]
results_table = results_table.loc[:, [*base_columns, *summary_columns]]
display(results_table)


,project_name,CSynthesisReport.BestLatency,CSynthesisReport.LUT,CSynthesisReport.FF,CSynthesisReport.DSP,CSynthesisReport.BRAM_18K
0,MNIST-Qkeras-hls4ml_prj_stream_latency,2367,143665,34445,46,134
1,MNIST-Qkeras-hls4ml_prj_base,2367,143665,34445,46,134
